# Notebook 15 (`15_digits_multiseed.ipynb`) — Validazione a 5 seed (source + adattamento), SVHN → MNIST/USPS

Ripete l'adattamento a 2 bracci (`shot_im`, `u_sfan` — **non** `epistemic_only`, che
`code_v2/src/digits_adapt.py` non espone: verificato prima di iniziare) già fatto nel
Notebook 12 (`12_digits_shift_adapt.ipynb`, che resta a singolo seed: training source
seed=2019, adattamento seed=1 per mnist e seed=2 per usps), ma su **5 seed indipendenti**
(`SEEDS = [0, 1, 2, 3, 4]`).

Qui il seed varia **sia il training del source model (SVHN) sia l'adattamento**: per ciascun
seed si riaddestra SVHN da zero, si rifitta la Laplace, si ricontrolla la convergenza MC, e
si riesegue l'adattamento a 2 bracci su entrambi i target con quello stesso seed (seed
condiviso fra i due bracci per un dato target, così che l'accoppiamento del Wilcoxon sia
legittimo).

**Riproducibilità del training.** `train_source_model` fissa
`torch.backends.cudnn.deterministic = True` e `cudnn.benchmark = False`, senza i quali i
soli seed non bastano su CUDA: due run allo stesso seed divergevano di ~3pp sull'accuracy
target (misurato a `seed=2019`: mnist 60.55% vs 57.13%), una deriva **dello stesso ordine
della deviazione standard fra seed che questo notebook vuole misurare**. Con quelle due
righe due run producono lo stesso `state_dict` bit-per-bit (verificato). Senza, una parte
della varianza riportata qui sotto sarebbe rumore di esecuzione anziché varianza di
inizializzazione.

**Il fit di Laplace gira sullo split di training di ciascun seed**, non sull'intero
`svhn_train.npz`: `theta_MAP` è il modo della cross-entropy sommata su *quei* punti più
`tau/2·||theta||²` con `tau = weight_decay × n_source_train`, quindi verosimiglianza e prior
devono stare sullo stesso N. Gli indici arrivano da `train_source_model` (vedi la Sezione 2
del Notebook 09).

**Nessuna duplicazione di codice**: `code_v2/src/digits_train.py::train_source_model(seed,
...)` (estratta da `main()`) e `code_v2/src/digits_adapt.py::adapt_target` (invariato). Il
fit di Laplace, il controllo di convergenza e la decomposizione BALD non esistono come
funzioni condivise per la pipeline digits: restano definiti localmente qui, nello stesso
stile inline dei Notebook 10, 11 e 12.

## 1. Stima del tempo totale, prima di lanciare i 5 seed per intero

Le costanti sotto vengono da tempi **misurati**, non da congetture, e sono quelle valide
con il percorso CUDA attivo (training, estrazione feature, adattamento e predittiva MC
girano tutti su GPU quando disponibile):

| voce | fonte | valore |
|---|---|---|
| training del source SVHN | `digits_train.py` eseguito dentro un notebook che tiene già stato | ~100 s |
| estrazione feature (4 domini, ~104k immagini) | misurato: 73.257 immagini in 2.8 s su CUDA (10.3 s su CPU) | ~5 s |
| fit di Laplace + sweep di convergenza MC | Notebook 10 con backend CUDA: 70 s per l'intero notebook | ~60 s |
| adattamento, 1 braccio su `mnist` (N=10.000) | misurato: 12.9 s (`shot_im`) / 36.6 s (`u_sfan`) per 50 step | ~25 s |
| adattamento, 1 braccio su `usps` (N=2.007) | scala con N | ~6 s |

**Perché queste cifre sono molto più basse di quelle di una versione precedente di questa
cella** (che stimava ~67 minuti e ne osservava ~104): la predittiva MC è passata da numpy
su CPU a torch su CUDA. Su forme reali quel calcolo è limitato dalla banda di memoria
(l'82% del tempo è softmax + entropia, non il prodotto matriciale), quindi la GPU rende
~64x; l'adattamento, che chiama la predittiva a ogni step, ne beneficia di riflesso.
`code_v2/src/bayesian_model.py` riporta le misure e il controllo di equivalenza fra i due
backend (concordano a ~1e-7 nat).

In [1]:
EST_TRAIN_S = 100        # digits_train.py dentro un notebook che tiene gia' stato (da solo: ~48 s)
EST_EXTRACT_S = 5        # 4 domini, ~104k immagini, extract su CUDA
EST_LAPLACE_S = 60       # fit + sweep di convergenza su 3 domini, backend CUDA (Notebook 10)
EST_MNIST_PER_ARM_S = 25    # 50 step full batch su N=10.000 (13 s shot_im, 37 s u_sfan)
EST_USPS_PER_ARM_S = 6      # 50 step full batch su N=2.007
N_ARMS = 2
N_SEEDS = 5

est_adapt_s = N_ARMS * (EST_MNIST_PER_ARM_S + EST_USPS_PER_ARM_S)
est_per_seed_s = EST_TRAIN_S + EST_EXTRACT_S + EST_LAPLACE_S + est_adapt_s
est_total_s = est_per_seed_s * N_SEEDS

print(f"stima per seed: training={EST_TRAIN_S}s + extract~{EST_EXTRACT_S}s + "
      f"laplace/convergenza~{EST_LAPLACE_S}s + adattamento({N_ARMS} bracci x (mnist+usps))"
      f"={est_adapt_s:.0f}s = {est_per_seed_s:.0f}s (~{est_per_seed_s/60:.1f} min)")
print(f"stima TOTALE per {N_SEEDS} seed: {est_total_s:.0f}s (~{est_total_s/60:.1f} minuti)")
print()
print("(stima PRIMA di lanciare la cella lunga sotto -- il tempo osservato viene")
print(" confrontato con questa stima subito dopo l'esecuzione)")

stima per seed: training=100s + extract~5s + laplace/convergenza~60s + adattamento(2 bracci x (mnist+usps))=62s = 227s (~3.8 min)
stima TOTALE per 5 seed: 1135s (~18.9 minuti)

(stima PRIMA di lanciare la cella lunga sotto -- il tempo osservato viene
 confrontato con questa stima subito dopo l'esecuzione)


## 2. Verifica di M_FIXED su più seed, prima di assumerlo fisso

Invece di assumere l'`M_FIXED` del seed 0, lo ricalcolo per ciascuno dei 5 (il costo del
controllo di convergenza, ~400 s stimati sopra, è comunque una frazione del totale).
**Anticipazione del risultato (Sezione 3): `M_FIXED` NON è stabile fra seed** — esce
500, 1000, 500, 1000, 500, cioè un fattore 2x fra il minimo e il massimo. Ricalcolarlo per
ogni seed si conferma necessario, non solo prudente.

## 3. Esecuzione dei 5 seed: training source + Laplace + convergenza + BALD + adattamento a 2 bracci x 2 target

Stesso protocollo di convergenza MC di `10_digits_bald.ipynb`/`12_digits_shift_adapt.ipynb`
(sweep `M_VALUES`, riferimento indipendente `M_REFERENCE=5000`, soglia relativa 1% + assoluta
2% del massimo osservato, finestra di stabilità di 3 valori consecutivi), `tau_prior =
weight_decay * n_source_train` (split interno di training di quel seed, non l'intero SVHN
train), `ADAPT_STEPS=50` (stessa convenzione di `12_digits_shift_adapt.ipynb`), stesso seed
condiviso fra i due bracci per un dato target in un dato seed (accoppiamento Wilcoxon
legittimo). Progresso stampato seed per seed (non solo alla fine), risultati salvati
incrementalmente dopo ogni seed.

In [2]:
import sys, time, copy
from pathlib import Path

here = Path().resolve()
for base in [here, *here.parents]:
    if (base / "code_v2" / "src" / "laplace_core.py").is_file():
        sys.path.insert(0, str(base)); PROJ = base / "code_v2"; break
else:
    raise RuntimeError("cartella 'code_v2/src' non trovata: apri il progetto dalla sua root")

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from code_v2.src.digits_train import train_source_model
from code_v2.src.digits_data import load_domain
from code_v2.src.bayesian_model import extract, augment, head_weights, LastLayerLaplace
from code_v2.src.digits_adapt import adapt_target

# Estrazione delle feature e adattamento girano sul device migliore disponibile.
# Il training del source lo fa gia' da se' (resolve_device in digits_train.py) e la
# predittiva MC sceglie il proprio backend (vedi bayesian_model.py). Misurato su
# questa macchina: extract su 73.257 immagini 10.3s -> 2.8s, e un passo di
# adapt_target su N=10.000 da 3.2-4.8s a 0.26-0.73s (6.5-12.6x).
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device per extract/adattamento: {DEVICE}")

SEEDS = [0, 1, 2, 3, 4]
TARGET_DOMAINS = ["mnist", "usps"]
EVAL_DOMAINS = ["svhn"] + TARGET_DOMAINS
ARM_WEIGHT_MODE = {"shot_im": "none", "u_sfan": "uncertainty"}
BASE_KWARGS = dict(gamma=0.5, temperature=0.4, lr=1e-2, M=100)
ADAPT_STEPS = 50

M_VALUES = [50, 100, 250, 500, 1000, 2000, 3000, 4000]
M_REFERENCE = 5000
RELATIVE_THRESHOLD = 0.01
ABSOLUTE_THRESHOLD_FRAC = 0.02
STABILITY_WINDOW = 3
CONVERGENCE_RNG_SEED = 123
FINAL_SEED = 456


def fit_laplace_and_check_convergence(model, Phi_aug_train, tau_prior, Phi_aug_eval, eval_domains):
    W_aug = head_weights(model)
    laplace = LastLayerLaplace.fit(W_aug, Phi_aug_train, tau_prior=tau_prior)

    convergence = {f"{d}_epi": [] for d in eval_domains}
    convergence["M"] = []
    for M in M_VALUES:
        convergence["M"].append(M)
        for domain in eval_domains:
            rng = np.random.default_rng(CONVERGENCE_RNG_SEED)
            pred = laplace.predictive_batched(Phi_aug_eval[domain], M=M, rng=rng)
            convergence[f"{domain}_epi"].append(pred["epistemic"].mean())

    ref_epi = {}
    for domain in eval_domains:
        rng = np.random.default_rng(CONVERGENCE_RNG_SEED)
        pred_ref = laplace.predictive_batched(Phi_aug_eval[domain], M=M_REFERENCE, rng=rng)
        ref_epi[domain] = pred_ref["epistemic"].mean()

    epi_max = max(list(ref_epi.values()) + sum([convergence[f"{d}_epi"] for d in eval_domains], []))
    absolute_threshold = ABSOLUTE_THRESHOLD_FRAC * epi_max

    def check_point(i):
        return all(abs(convergence[f"{d}_epi"][i] - ref_epi[d]) / ref_epi[d] < RELATIVE_THRESHOLD
                   and abs(convergence[f"{d}_epi"][i] - ref_epi[d]) < absolute_threshold
                   for d in eval_domains)

    point_ok = [check_point(i) for i in range(len(convergence["M"]))]
    M_FIXED = None
    for i, M in enumerate(convergence["M"]):
        if i + STABILITY_WINDOW <= len(convergence["M"]) and all(point_ok[i:i + STABILITY_WINDOW]):
            M_FIXED = M
            break
    if M_FIXED is None:
        M_FIXED = M_REFERENCE
    return laplace, M_FIXED


def compute_predictive_results(laplace, Phi_aug_eval, y_eval, M_FIXED, seed=FINAL_SEED):
    predictive_results = {}
    for domain, Phi_aug in Phi_aug_eval.items():
        rng = np.random.default_rng(seed)
        pred = laplace.predictive_batched(Phi_aug, M=M_FIXED, rng=rng)
        predictive_results[domain] = dict(y=y_eval[domain], **pred)
    return predictive_results


results_per_seed = {}
t_all0 = time.time()
for seed in SEEDS:
    t_seed0 = time.time()
    print(f"\n{'#'*70}\n# SEED {seed}\n{'#'*70}")

    t0 = time.time()
    res = train_source_model(seed=seed, verbose=False)
    model, mean, std = res["model"], res["mean"], res["std"]
    model.eval()
    n_source_train, weight_decay = res["n_source_train"], res["weight_decay"]
    print(f"  train_source_model: {time.time()-t0:.1f}s  source_test_acc={100*res['source_test_acc']:.2f}%  "
          f"target_test_acc={ {k: round(100*v,2) for k,v in res['target_test_acc'].items()} }")

    t0 = time.time()
    # Il fit di Laplace gira sullo split di training di QUESTO seed (indici
    # restituiti da train_source_model), non sull'intero svhn_train: theta_MAP e'
    # il modo della CE sommata su quei punti piu' tau/2*||theta||^2 con
    # tau = weight_decay * n_source_train, quindi verosimiglianza e prior devono
    # stare sullo stesso N. Vedi la sezione 2 di 09_digits_bayesian_source.ipynb.
    X_train_full, y_train_full = load_domain("svhn", "train", mean, std)
    train_idx = res["train_indices"]
    X_svhn_train, y_svhn_train = X_train_full[train_idx], y_train_full[train_idx]
    assert X_svhn_train.shape[0] == n_source_train
    train_loader = DataLoader(TensorDataset(X_svhn_train, y_svhn_train), batch_size=256, shuffle=False)
    Phi_train, _, _ = extract(model, train_loader, device=DEVICE)
    Phi_aug_train = augment(Phi_train)
    tau_prior = weight_decay * n_source_train

    Phi_aug_eval, y_eval = {}, {}
    for domain in EVAL_DOMAINS:
        X_d, y_d = load_domain(domain, "test", mean, std)
        loader = DataLoader(TensorDataset(X_d, y_d), batch_size=256, shuffle=False)
        Phi_d, y_d_np, _ = extract(model, loader, device=DEVICE)
        Phi_aug_eval[domain] = augment(Phi_d)
        y_eval[domain] = y_d_np

    laplace, M_FIXED = fit_laplace_and_check_convergence(model, Phi_aug_train, tau_prior, Phi_aug_eval, EVAL_DOMAINS)
    print(f"  laplace+convergence: {time.time()-t0:.1f}s  M_FIXED={M_FIXED}")

    predictive_results = compute_predictive_results(laplace, Phi_aug_eval, y_eval, M_FIXED)
    ratios = {}
    for d in EVAL_DOMAINS:
        r = predictive_results[d]
        alea, epi = r["aleatoric"].mean(), r["epistemic"].mean()
        ratios[d] = dict(alea=float(alea), epi=float(epi), ratio=float(alea/epi))
        print(f"    {d}: alea={alea:.4f} epi={epi:.4f} ratio={alea/epi:.2f}x")

    target_raw = {d: load_domain(d, "test", mean, std) for d in TARGET_DOMAINS}
    adaptation = {}
    for domain in TARGET_DOMAINS:
        t0 = time.time()
        X_t, y_t = target_raw[domain]
        X_t = X_t.to(DEVICE)
        acc_pre = (predictive_results[domain]["probs"].argmax(axis=1) == y_eval[domain]).mean()
        adaptation[domain] = {}
        for arm, weight_mode in ARM_WEIGHT_MODE.items():
            m = copy.deepcopy(model).to(DEVICE)
            hist = adapt_target(m, laplace, X_t, weight_mode=weight_mode, steps=ADAPT_STEPS,
                                seed=seed, **BASE_KWARGS)
            m.eval()
            with torch.no_grad():
                probs_post = torch.softmax(m(X_t), dim=-1).cpu().numpy()
            acc_post = (probs_post.argmax(axis=1) == y_t.numpy()).mean()
            adaptation[domain][arm] = dict(acc_pre=float(acc_pre), acc_post=float(acc_post))
            print(f"    {domain}/{arm} (seed={seed}): pre={100*acc_pre:.2f}%  post={100*acc_post:.2f}%  "
                  f"delta={100*(acc_post-acc_pre):+.2f}pp")
        print(f"  {domain} adaptation (2 arms): {time.time()-t0:.1f}s")

    results_per_seed[seed] = dict(
        source_test_acc=res["source_test_acc"], target_test_acc=res["target_test_acc"],
        M_FIXED=M_FIXED, ratios=ratios, adaptation=adaptation,
    )
    print(f"  TOTALE SEED {seed}: {time.time()-t_seed0:.1f}s")

print(f"\nTOTALE COMPLESSIVO: {time.time()-t_all0:.1f}s")

device per extract/adattamento: cuda

######################################################################
# SEED 0
######################################################################


  train_source_model: 65.6s  source_test_acc=87.30%  target_test_acc={'mnist': 65.26, 'usps': 64.03}


  laplace+convergence: 63.7s  M_FIXED=500


    svhn: alea=0.3633 epi=0.0053 ratio=68.41x
    mnist: alea=0.2564 epi=0.0321 ratio=8.00x
    usps: alea=0.3322 epi=0.0213 ratio=15.58x


    mnist/shot_im (seed=0): pre=65.22%  post=75.15%  delta=+9.93pp


...

    svhn: alea=0.2955 epi=0.0054 ratio=54.88x
    mnist: alea=0.2032 epi=0.0380 ratio=5.35x
    usps: alea=0.3086 epi=0.0232 ratio=13.30x


    mnist/shot_im (seed=4): pre=57.28%  post=74.15%  delta=+16.87pp


    mnist/u_sfan (seed=4): pre=57.28%  post=77.29%  delta=+20.01pp
  mnist adaptation (2 arms): 184.0s


    usps/shot_im (seed=4): pre=59.44%  post=72.65%  delta=+13.20pp


    usps/u_sfan (seed=4): pre=59.44%  post=66.77%  delta=+7.32pp
  usps adaptation (2 arms): 5.2s
  TOTALE SEED 4: 338.4s

TOTALE COMPLESSIVO: 1640.8s


**Tempo effettivo: 1640.8 s (~27 minuti), contro una stima di 1135 s (~19 minuti).** La
stima sbaglia per difetto di ~1.4x, un errore molto più contenuto di quello della versione
CPU di questo notebook (che stimava 67 minuti e ne osservava 104).

| voce | stima | osservato (per seed) |
|---|---:|---:|
| training del source | 100 s | 65.6 – 78.4 s |
| fit di Laplace + sweep di convergenza (+ extract) | 65 s | 63.7 – 69.0 s |
| adattamento (2 bracci × 2 target) | 62 s | ~180 – 190 s (per differenza) |
| **totale per seed** | **227 s** | **320.5 – 338.4 s** |

Training e fit di Laplace sono ora stimati bene; **l'adattamento resta sottostimato di ~3x**.
La ragione è che le misure per braccio (12.9 s per `shot_im`, 36.6 s per `u_sfan` su
`mnist`) erano prese su un processo appena avviato e con un solo modello in memoria,
mentre qui il ciclo tiene contemporaneamente il training set SVHN normalizzato, le feature
di tutti e tre i domini e le copie del modello per i due bracci. È lo stesso effetto già
osservato sul training nella versione CPU: **un tempo misurato in isolamento non si
trasferisce a un ciclo che tiene stato**, e la direzione dell'errore è sempre la stessa.

**Confronto con il percorso numpy.** Lo stesso notebook, prima di spostare predittiva MC,
estrazione feature e adattamento su CUDA, impiegava **6267.8 s (104 minuti)**: 3.8x più
lento. I risultati non cambiano — i rapporti aleatoria/epistemica e gli `M_FIXED` per seed
sono identici fra le due esecuzioni (si vedano le Sezioni 4 e 2); differiscono solo i delta
di accuratezza dell'adattamento, di ~1-2pp, perché l'adattamento è un'ottimizzazione a 50
passi in cui la differenza float32/float64 si accumula.

## 4. Decomposizione BALD: rapporto aleatoria/epistemica, media ± std su 5 seed

In [3]:
DOMAINS_BALD = ["svhn", "mnist", "usps"]
print(f"{'dominio':>8s} {'media':>10s} {'std':>8s} {'valori (uno per seed)'}")
print("-" * 60)
for d in DOMAINS_BALD:
    vals = [results_per_seed[s]["ratios"][d]["ratio"] for s in SEEDS]
    print(f"{d:>8s} {np.mean(vals):9.2f}x {np.std(vals, ddof=1):7.2f}x   {[round(v,2) for v in vals]}")

print(f"\nM_FIXED per seed: { {s: results_per_seed[s]['M_FIXED'] for s in SEEDS} }")

 dominio      media      std valori (uno per seed)
------------------------------------------------------------
    svhn     67.57x   10.08x   [68.41, 63.77, 68.07, 82.74, 54.88]
   mnist      6.42x    1.05x   [8.0, 6.1, 5.76, 6.89, 5.35]
    usps     14.37x    1.45x   [15.58, 13.71, 13.02, 16.24, 13.3]

M_FIXED per seed: {0: 500, 1: 1000, 2: 500, 3: 1000, 4: 500}


**Il rapporto sul source (SVHN) è 67.6x ± 10.1x**, coerente con i 57x misurati sul
singolo seed del Notebook 10 (che rientra vicino al bordo basso del range osservato qui,
54.9 – 82.7x). Il source resta quindi saldamente aleatoria-dominante in ogni run.

Sui target il rapporto è molto più basso: `mnist` 6.4x ± 1.1x, `usps` 14.4x ± 1.5x, entrambi
ben sotto quello del source. La lettura è che **l'epistemica cresce sui target molto più di
quanto cali l'aleatoria**: passando da svhn a mnist l'epistemica media va da ~0.005 a ~0.037
(×7), mentre l'aleatoria scende solo da ~0.35 a ~0.23. È esattamente il comportamento che ci
si aspetta da un rilevatore di shift — ma resta vero che, in valore assoluto, l'epistemica è
un ordine di grandezza sotto l'aleatoria anche sui target, ed è per questo che pesare per
l'entropia **totale** (U-SFAN) non riesce a isolarla.

La deviazione standard sul source (10.1 su 67.6, ~15% della media) è contenuta: il *regime*
di incertezza è riproducibile fra seed, anche quando l'esito dell'adattamento non lo è.

## 5. Tabelle (una per target): accuracy pre/post/delta, media ± std sui 2 bracci

In [4]:
ARMS = ["shot_im", "u_sfan"]
TARGETS = ["mnist", "usps"]

summary = {}
for t in TARGETS:
    summary[t] = {}
    print(f"=== target: {t} ===")
    print(f"{'braccio':>10s} {'pre':>16s} {'post':>16s} {'delta':>18s}")
    print("-" * 64)
    for a in ARMS:
        pre = np.array([results_per_seed[s]["adaptation"][t][a]["acc_pre"] for s in SEEDS])
        post = np.array([results_per_seed[s]["adaptation"][t][a]["acc_post"] for s in SEEDS])
        delta = post - pre
        summary[t][a] = dict(pre=pre, post=post, delta=delta)
        print(f"{a:>10s} {100*pre.mean():6.2f}%+-{100*pre.std(ddof=1):4.2f} "
              f"{100*post.mean():6.2f}%+-{100*post.std(ddof=1):4.2f} "
              f"{100*delta.mean():+7.2f}pp+-{100*delta.std(ddof=1):5.2f}")
    print()

=== target: mnist ===
   braccio              pre             post              delta
----------------------------------------------------------------
   shot_im  60.18%+-3.15  74.43%+-5.04  +14.25pp+- 4.83
    u_sfan  60.18%+-3.15  72.23%+-4.10  +12.04pp+- 5.59

=== target: usps ===
   braccio              pre             post              delta
----------------------------------------------------------------
   shot_im  60.25%+-2.37  49.47%+-21.65  -10.78pp+-21.14
    u_sfan  60.25%+-2.37  51.34%+-11.97   -8.91pp+-11.65



**Su `usps` la deviazione standard supera in valore assoluto la media** (21.14pp e
11.65pp di std contro delta medi di −10.78pp e −8.91pp). Peggio: **il delta medio è
negativo per entrambi i bracci** — su questo target, con questi iperparametri
(`lr=1e-2`, `steps=50`, full batch), l'adattamento in media *danneggia* il modello.

Il dettaglio per seed (Sezione 3) mostra da dove viene la varianza: `shot_im` va da
+13.20pp (seed 4) a **−42.45pp** (seed 2), `u_sfan` da +7.32pp (seed 4) a **−20.73pp**
(seed 1). Non è rumore intorno a una media: sono collassi, in cui l'ottimizzazione IM porta
il modello in una regione da cui non torna. Su `usps` (N=2.007) il termine di diversità ha
molti meno campioni per stabilizzare la distribuzione di batch che su `mnist` (N=10.000),
il che rende il collasso su una o poche classi molto più facile.

Su `mnist` il quadro è più sano — entrambi i bracci migliorano in media (+14.25pp e
+12.04pp) — ma la std resta ~5pp, quindi anche lì un singolo run non è rappresentativo.

## 6. Wilcoxon signed-rank, accoppiato per seed

In [5]:
from scipy.stats import wilcoxon

for t in TARGETS:
    d_shot, d_usfan = summary[t]["shot_im"]["delta"], summary[t]["u_sfan"]["delta"]
    stat, p = wilcoxon(d_shot, d_usfan)
    print(f"=== {t} ===")
    print(f"  shot_im vs u_sfan:          W={stat:.1f}  p={p:.4f}")
    print(f"    differenze (pp), una per seed: {[round(100*x,2) for x in (d_shot-d_usfan)]}")
    print()

=== mnist ===
  shot_im vs u_sfan:          W=7.0  p=1.0000
    differenze (pp), una per seed: [np.float64(1.31), np.float64(-1.08), np.float64(-2.35), np.float64(16.27), np.float64(-3.14)]

=== usps ===
  shot_im vs u_sfan:          W=7.0  p=1.0000
    differenze (pp), una per seed: [np.float64(3.69), np.float64(23.12), np.float64(-26.36), np.float64(-15.7), np.float64(5.88)]



**Nessuno dei due confronti distingue i bracci: `p = 1.0000` su entrambi i target.**
È il valore massimo possibile del test: la somma dei ranghi positivi e negativi è
esattamente bilanciata. Su `mnist` 2/5 seed favoriscono `shot_im` e 3/5 `u_sfan`; su `usps`
3/5 favoriscono `shot_im` e 2/5 `u_sfan`, con magnitudini che si annullano.

Le differenze per seed lo mostrano bene: su `usps` vanno da **+23.12pp a −26.36pp**, cioè in
una run `shot_im` batte `u_sfan` di 23 punti e in un'altra perde di 26. Non c'è un effetto
da misurare; c'è un processo di ottimizzazione instabile di cui stiamo campionando la
distribuzione. Anche su `mnist`, dove le differenze sono più contenute, un singolo seed
(il 3) vale +16.27pp mentre gli altri quattro stanno entro ±3.2pp.

> **Nota sulla potenza del test.** Con n=5 il p-value **minimo ottenibile** dal Wilcoxon
> signed-rank a due code è 0.0625: nessuno di questi confronti *poteva* risultare
> significativo a 0.05, comunque fossero andati i dati. Servono almeno 6 seed perché la
> soglia convenzionale sia raggiungibile. Qui però il risultato è netto a prescindere:
> `p = 1.0` non è "non abbiamo abbastanza potenza", è "i due bracci sono indistinguibili
> su questo campione".

## 7. Confronto esplicito: singolo seed (`12_digits_shift_adapt.ipynb`) vs. media 5 seed

| target | delta shot_im (1 seed) | delta u_sfan (1 seed) | delta shot_im (media 5 seed) | delta u_sfan (media 5 seed) | pattern confermato? | p Wilcoxon |
|---|---|---|---|---|---|---|
| mnist | +6.49pp | +1.42pp | +14.25pp ± 4.83 | +12.04pp ± 5.59 | **no** — il divario di 5.1pp del singolo seed scende a 2.2pp e cambia segno in 3/5 run | 1.000 |
| usps | +7.18pp | +10.27pp | **−10.78pp ± 21.14** | **−8.91pp ± 11.65** | **no** — entrambi i bracci in media *danneggiano* il modello | 1.000 |

**Il singolo seed dava un quadro sostanzialmente più roseo del reale su entrambi i target,
ma per ragioni diverse.**

Su **mnist** il seed 2019 mostrava un divario largo a favore di `shot_im` (+6.49 contro
+1.42pp). Sui 5 seed quel divario si riduce (+14.25 contro +12.04pp) e il segno si inverte
in 3 run su 5: le differenze per seed sono `[+1.31, −1.08, −2.35, +16.27, −3.14]`pp, cioè
quattro run entro ±3.2pp e una singola run che da sola sposta la media. Non c'era un
effetto da misurare, c'era un campione di una distribuzione larga.

Su **usps** il problema è più grave di un'inversione di ordine fra i bracci: sul seed 2019
l'adattamento aiutava entrambi (+7.18 e +10.27pp), mentre sui 5 seed la media è **negativa
per entrambi** (−10.78 e −8.91pp), con casi singoli fino a −42.45pp (`shot_im`, seed 2) e
−20.73pp (`u_sfan`, seed 1). Il seed originariamente riportato era fra i più fortunati, e
il rischio di collasso era **completamente invisibile** in quell'unica run.

**La lezione operativa**: su questo esperimento un risultato a singolo seed non porta
informazione utilizzabile né sull'ordine dei bracci né sul *segno* del beneficio
dell'adattamento.

## 8. Collocazione nel confronto multi-esperimento, e aggiornamento della discussione di letteratura

| esperimento | rapporto alea/epi (source) | vince (1 seed) | vince (media 5 seed) |
|---|---|---|---|
| SVHN → MNIST/USPS (questo notebook) | 67.6x ± 10.1x | shot_im su mnist, u_sfan su usps | **nessuno** — `p = 1.0` su entrambi i target |
| MNIST → Rotated-MNIST 30° (Notebook 07) | 13.3x ± 1.9x (clean) | u_sfan | **u_sfan**, 4/5 seed |
| Electronics → dvd/kitchen/books (Amazon Reviews) | ~95.9x – 99.7x | u_sfan su tutti e 3 | u_sfan su tutti e 3, confermato |

**Aggiornamento della discussione di Kendall & Gal (2017).** Il Notebook 12 interpretava il
vantaggio di `shot_im` su SVHN come conferma diretta del meccanismo: un source con poca
epistemica residua rende controproducente il peso a entropia totale di U-SFAN, che finisce
per pesare l'aleatoria scambiata per incertezza. **Il risultato multi-seed non smentisce il
meccanismo, ma toglie a questo esperimento il ruolo di sua prova.** Il rapporto sul source
resta alto e stabile (67.6x ± 10.1x, ~15% di variabilità relativa), quindi il regime di
incertezza *è* riproducibile fra seed; ciò che non è riproducibile è l'**esito
dell'adattamento**. Con `p = 1.0` su entrambi i target, questo esperimento semplicemente
non discrimina fra i due bracci.

**Dove la prova regge invece è il Notebook 07** (MNIST → Rotated-MNIST): là il rapporto
scende a ~13x, `u_sfan` vince in 4 seed su 5 con margine consistente e le deviazioni
standard sui delta sono di un ordine di grandezza più piccole (~1.5pp contro 5-21pp qui).
È quello, non questo, il supporto empirico all'ipotesi "il vantaggio di U-SFAN dipende dal
regime di incertezza del source".

**Che cosa distingue gli esperimenti che reggono da questo.** Non il valore del rapporto:
Amazon Reviews ha un rapporto ancora più estremo (~96-100x) e si conferma. La differenza
sembra stare nella **stabilità dell'ottimizzazione**, non in quella del regime di
incertezza. Qui l'adattamento su `usps` collassa in 3 run su 5 (fino a −42.45pp), e la
varianza dell'esito (std 12-21pp) sommerge qualunque differenza fra bracci (~2pp). Un
metodo di adattamento che collassa in metà delle run non può essere confrontato con un
altro finché il collasso non è sotto controllo — `lr`, numero di step e la dimensione del
batch su cui si calcola il termine di diversità (2.007 campioni per `usps`) sono i sospetti
naturali. È un'ipotesi da testare, non una conclusione.

**Nota conclusiva.** Il verdetto "il pattern a singolo seed regge?" è ora **sì** per due
esperimenti (Rotated-MNIST, Amazon Reviews) e **no** per questo. Non esiste quindi
un'aspettativa di default applicabile a questa classe di esperimenti (CNN piccola, IM
adaptation full-batch): ogni pattern a singolo seed va verificato caso per caso, non per
analogia.